In [ ]:
!pip install tensorflow numpy pandas scikit-learn

In [ ]:
"""
Intelligent Clinical Decision Support System for Thoracic Pathology Diagnosis
Using NIH Chest X-ray 14 Dataset with TensorFlow

Properly organized and structured code for pre-divided datasets
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
import numpy as np
import pandas as pd
import os
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from datetime import datetime

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


class PreDividedDataLoader:
    """
    Data loader for NIH Chest X-ray dataset already divided into train/val/test folders
    """
    
    def __init__(self, base_dir, labels_csv, image_size=(224, 224), batch_size=32):
        """
        Initialize the data loader
        
        Args:
            base_dir: Base directory containing train/, val/, test/ folders
            labels_csv: Path to CSV file with all image labels
            image_size: Target image size (height, width)
            batch_size: Batch size for training
        """
        self.base_dir = base_dir
        self.train_dir = os.path.join(base_dir, "train")
        self.val_dir = os.path.join(base_dir, "val")
        self.test_dir = os.path.join(base_dir, "test")
        self.labels_csv = labels_csv
        self.image_size = image_size
        self.batch_size = batch_size
        
        # Define the 14 pathology classes
        self.pathologies = [
            'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
            'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
            'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
            'Pleural_Thickening', 'Hernia'
        ]
        
        self.num_classes = len(self.pathologies)
        
        # Verify directories exist
        self._verify_directories()
        
        # Load and prepare labels
        self._load_labels()
    
    def _verify_directories(self):
        """Verify that all required directories exist"""
        print("=" * 80)
        print("VERIFYING DIRECTORIES")
        print("=" * 80)
        
        for name, path in [("Train", self.train_dir), 
                          ("Validation", self.val_dir), 
                          ("Test", self.test_dir)]:
            if not os.path.exists(path):
                raise FileNotFoundError(f"{name} directory not found: {path}")
            
            num_files = len([f for f in os.listdir(path) 
                           if os.path.isfile(os.path.join(path, f))])
            print(f"✓ {name:12s} directory: {num_files:6d} images at {path}")
    
    def _load_labels(self):
        """Load CSV and create binary labels for each pathology"""
        print(f"\n{'=' * 80}")
        print("LOADING LABELS")
        print("=" * 80)
        
        print(f"Loading from: {self.labels_csv}")
        self.df = pd.read_csv(self.labels_csv)
        print(f"✓ Loaded {len(self.df)} image labels from CSV")
        
        # Create binary columns for each pathology
        print("\nCreating binary labels for each pathology...")
        for pathology in self.pathologies:
            self.df[pathology] = self.df['Finding Labels'].apply(
                lambda x: 1 if pathology in str(x) else 0
            )
        
        # Display label distribution
        print("\nLabel Distribution:")
        print("-" * 80)
        for pathology in self.pathologies:
            count = self.df[pathology].sum()
            percentage = (count / len(self.df)) * 100
            print(f"{pathology:20s}: {count:6d} ({percentage:5.2f}%)")
    
    def create_generators(self):
        """
        Create data generators for train, validation, and test sets
        
        Returns:
            train_generator, val_generator, test_generator
        """
        print(f"\n{'=' * 80}")
        print("CREATING DATA GENERATORS")
        print("=" * 80)
        
        # Get image filenames from each directory
        train_images = set(os.listdir(self.train_dir))
        val_images = set(os.listdir(self.val_dir))
        test_images = set(os.listdir(self.test_dir))
        
        print(f"\nImages in directories:")
        print(f"  Train:      {len(train_images)}")
        print(f"  Validation: {len(val_images)}")
        print(f"  Test:       {len(test_images)}")
        
        # Create DataFrames for each split based on folder location
        train_df = self.df[self.df['Image Index'].isin(train_images)].copy()
        val_df = self.df[self.df['Image Index'].isin(val_images)].copy()
        test_df = self.df[self.df['Image Index'].isin(test_images)].copy()
        
        print(f"\nMatched with CSV labels:")
        print(f"  Train:      {len(train_df)} images")
        print(f"  Validation: {len(val_df)} images")
        print(f"  Test:       {len(test_df)} images")
        
        # Data augmentation for training
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1,
            fill_mode='nearest'
        )
        
        # Only rescaling for validation and test
        val_test_datagen = ImageDataGenerator(rescale=1./255)
        
        # Create generators
        print("\nGenerating data loaders...")
        
        train_generator = train_datagen.flow_from_dataframe(
            dataframe=train_df,
            directory=self.train_dir,
            x_col='Image Index',
            y_col=self.pathologies,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='raw',
            shuffle=True
        )
        
        val_generator = val_test_datagen.flow_from_dataframe(
            dataframe=val_df,
            directory=self.val_dir,
            x_col='Image Index',
            y_col=self.pathologies,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='raw',
            shuffle=False
        )
        
        test_generator = val_test_datagen.flow_from_dataframe(
            dataframe=test_df,
            directory=self.test_dir,
            x_col='Image Index',
            y_col=self.pathologies,
            target_size=self.image_size,
            batch_size=self.batch_size,
            class_mode='raw',
            shuffle=False
        )
        
        print(f"\n✓ Training generator:   {len(train_generator)} batches")
        print(f"✓ Validation generator: {len(val_generator)} batches")
        print(f"✓ Test generator:       {len(test_generator)} batches")
        
        # Verify generators work
        self._verify_generator(train_generator)
        
        return train_generator, val_generator, test_generator
    
    def _verify_generator(self, generator):
        """Test that generator produces valid batches"""
        print("\nVerifying generator output...")
        try:
            x_batch, y_batch = next(generator)
            print(f"✓ Image batch shape: {x_batch.shape}")
            print(f"✓ Label batch shape: {y_batch.shape}")
            print(f"✓ Image value range: [{x_batch.min():.3f}, {x_batch.max():.3f}]")
            print(f"✓ Sample has {int(y_batch[0].sum())} positive pathologies")
        except Exception as e:
            print(f"❌ Generator verification failed: {e}")
            raise


class ClinicalDecisionSupportModel:
    """Deep learning model for thoracic pathology diagnosis"""
    
    def __init__(self, num_classes=14, image_size=(224, 224), 
                 backbone='DenseNet121', model_dir='models', log_dir='logs'):
        """
        Initialize the model
        
        Args:
            num_classes: Number of pathology classes (14 for NIH dataset)
            image_size: Input image size
            backbone: Backbone architecture ('DenseNet121', 'ResNet50', 'EfficientNetB0')
            model_dir: Directory to save model checkpoints
            log_dir: Directory for TensorBoard logs
        """
        self.num_classes = num_classes
        self.image_size = image_size + (3,)  # Add channel dimension
        self.backbone = backbone
        self.model_dir = model_dir
        self.log_dir = log_dir
        self.model = None
        self.history = None
        
        # Create directories
        os.makedirs(model_dir, exist_ok=True)
        os.makedirs(log_dir, exist_ok=True)
    
    def build_model(self):
        """Build the deep learning model with transfer learning"""
        print(f"\n{'=' * 80}")
        print("BUILDING MODEL")
        print("=" * 80)
        print(f"Backbone: {self.backbone}")
        print(f"Input shape: {self.image_size}")
        print(f"Number of classes: {self.num_classes}")
        
        # Select backbone architecture
        if self.backbone == 'DenseNet121':
            base_model = DenseNet121(
                weights='imagenet',
                include_top=False,
                input_shape=self.image_size
            )
        elif self.backbone == 'ResNet50':
            base_model = ResNet50(
                weights='imagenet',
                include_top=False,
                input_shape=self.image_size
            )
        elif self.backbone == 'EfficientNetB0':
            base_model = EfficientNetB0(
                weights='imagenet',
                include_top=False,
                input_shape=self.image_size
            )
        else:
            raise ValueError(f"Unsupported backbone: {self.backbone}")
        
        # Freeze base model for transfer learning
        base_model.trainable = False
        
        # Build complete model
        inputs = keras.Input(shape=self.image_size)
        x = base_model(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        
        # Classification head
        x = layers.Dense(512, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        
        # Output layer - sigmoid for multi-label classification
        outputs = layers.Dense(self.num_classes, activation='sigmoid')(x)
        
        self.model = keras.Model(inputs=inputs, outputs=outputs, 
                                name=f'ChestXray_{self.backbone}')
        
        print(f"✓ Model built successfully")
        return self.model
    
    def compile_model(self, learning_rate=1e-4):
        """Compile the model"""
        print(f"\nCompiling model with learning rate: {learning_rate}")
        
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='binary_crossentropy',
            metrics=[
                'binary_accuracy',
                keras.metrics.AUC(name='auc', multi_label=True),
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')
            ]
        )
        
        print("✓ Model compiled successfully")
        return self.model
    
    def get_callbacks(self):
        """Create training callbacks"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        callbacks = [
            ModelCheckpoint(
                filepath=os.path.join(self.model_dir, f'best_model_{timestamp}.h5'),
                monitor='val_auc',
                mode='max',
                save_best_only=True,
                verbose=1
            ),
            EarlyStopping(
                monitor='val_auc',
                mode='max',
                patience=10,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            ),
            TensorBoard(
                log_dir=os.path.join(self.log_dir, timestamp),
                histogram_freq=1
            )
        ]
        
        print(f"\n✓ Callbacks configured")
        print(f"  Model checkpoints: {self.model_dir}")
        print(f"  TensorBoard logs:  {os.path.join(self.log_dir, timestamp)}")
        
        return callbacks
    
    def train(self, train_generator, val_generator, epochs=30, fine_tune_epochs=20):
        """
        Train model with two-phase approach
        
        Args:
            train_generator: Training data generator
            val_generator: Validation data generator
            epochs: Epochs for phase 1 (frozen base)
            fine_tune_epochs: Epochs for phase 2 (fine-tuning)
        """
        
        # Phase 1: Train top layers only
        print(f"\n{'=' * 80}")
        print("PHASE 1: TRAINING TOP LAYERS (BASE FROZEN)")
        print("=" * 80)
        print(f"Epochs: {epochs}")
        
        callbacks = self.get_callbacks()
        
        history1 = self.model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        # Phase 2: Fine-tune entire network
        print(f"\n{'=' * 80}")
        print("PHASE 2: FINE-TUNING ENTIRE NETWORK")
        print("=" * 80)
        print(f"Epochs: {fine_tune_epochs}")
        
        # Unfreeze base model
        self.model.layers[1].trainable = True
        
        # Freeze early layers
        for layer in self.model.layers[1].layers[:100]:
            layer.trainable = False
        
        # Recompile with lower learning rate
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-5),
            loss='binary_crossentropy',
            metrics=[
                'binary_accuracy',
                keras.metrics.AUC(name='auc', multi_label=True),
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')
            ]
        )
        
        callbacks = self.get_callbacks()
        
        history2 = self.model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=fine_tune_epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        # Combine histories
        self.history = {
            'loss': history1.history['loss'] + history2.history['loss'],
            'val_loss': history1.history['val_loss'] + history2.history['val_loss'],
            'auc': history1.history['auc'] + history2.history['auc'],
            'val_auc': history1.history['val_auc'] + history2.history['val_auc']
        }
        
        print("\n✓ Training completed successfully")
        return self.history
    
    def evaluate(self, test_generator, pathology_names):
        """Evaluate model on test set"""
        print(f"\n{'=' * 80}")
        print("EVALUATING MODEL ON TEST SET")
        print("=" * 80)
        
        # Get predictions
        predictions = self.model.predict(test_generator, verbose=1)
        
        # Get true labels
        y_true = np.vstack([test_generator[i][1] for i in range(len(test_generator))])
        
        # Calculate metrics
        results = {
            'overall_auc': roc_auc_score(y_true, predictions, average='macro'),
            'per_class_auc': {}
        }
        
        for i, pathology in enumerate(pathology_names):
            try:
                auc = roc_auc_score(y_true[:, i], predictions[:, i])
                results['per_class_auc'][pathology] = float(auc)
            except:
                results['per_class_auc'][pathology] = None
        
        # Print results
        print(f"\nOverall AUC: {results['overall_auc']:.4f}\n")
        print("Per-Class AUC Scores:")
        print("-" * 80)
        for pathology, auc in results['per_class_auc'].items():
            if auc is not None:
                print(f"{pathology:20s}: {auc:.4f}")
        
        return results, predictions, y_true
    
    def plot_training_history(self, save_path='training_history.png'):
        """Plot and save training history"""
        if self.history is None:
            print("No training history available")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        axes[0].plot(self.history['loss'], label='Training Loss')
        axes[0].plot(self.history['val_loss'], label='Validation Loss')
        axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # AUC
        axes[1].plot(self.history['auc'], label='Training AUC')
        axes[1].plot(self.history['val_auc'], label='Validation AUC')
        axes[1].set_title('Model AUC', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('AUC')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Training history plot saved to {save_path}")
        
        return fig
    
    def plot_auc_scores(self, results, save_path='auc_scores.png'):
        """Plot per-class AUC scores"""
        pathologies = [p for p in results['per_class_auc'].keys() 
                      if results['per_class_auc'][p] is not None]
        aucs = [results['per_class_auc'][p] for p in pathologies]
        
        plt.figure(figsize=(12, 6))
        bars = plt.barh(pathologies, aucs, color='steelblue')
        
        # Color by performance
        for i, bar in enumerate(bars):
            if aucs[i] >= 0.9:
                bar.set_color('green')
            elif aucs[i] >= 0.8:
                bar.set_color('steelblue')
            else:
                bar.set_color('orange')
        
        plt.xlabel('AUC Score', fontsize=12)
        plt.title('Per-Class AUC Scores - Thoracic Pathology Detection', 
                 fontsize=14, fontweight='bold')
        plt.xlim(0, 1)
        plt.axvline(x=0.8, color='red', linestyle='--', alpha=0.3, label='0.8 threshold')
        plt.axvline(x=0.9, color='green', linestyle='--', alpha=0.3, label='0.9 threshold')
        plt.legend()
        plt.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ AUC scores plot saved to {save_path}")
        
        return plt.gcf()
    
    def save_model(self, filepath='final_model.h5'):
        """Save the trained model"""
        self.model.save(filepath)
        print(f"✓ Model saved to {filepath}")


def main():
    """Main execution function"""
    
    print("=" * 80)
    print("INTELLIGENT CLINICAL DECISION SUPPORT SYSTEM")
    print("Thoracic Pathology Diagnosis from Chest X-rays")
    print("=" * 80)
    
    # ========================================================================
    # CONFIGURATION - UPDATE THESE PATHS
    # ========================================================================
    
    BASE_DIR = r"C:\Users\Phoenix\Downloads\FYB 26\NIH Chest X-ray\DATASET\Dataset"
    LABELS_CSV = r"C:\Users\Phoenix\Downloads\FYB 26\NIH Chest X-ray\DATASET\Data_Entry_2017.csv"
    
    IMAGE_SIZE = (224, 224)
    BATCH_SIZE = 32
    EPOCHS_PHASE1 = 30
    EPOCHS_PHASE2 = 20
    BACKBONE = 'DenseNet121'  # Options: 'DenseNet121', 'ResNet50', 'EfficientNetB0'
    
    # ========================================================================
    # STEP 1: LOAD DATA
    # ========================================================================
    
    print(f"\n{'STEP 1: LOADING DATA':^80}")
    print("=" * 80)
    
    data_loader = PreDividedDataLoader(
        base_dir=BASE_DIR,
        labels_csv=LABELS_CSV,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )
    
    train_gen, val_gen, test_gen = data_loader.create_generators()
    
    # ========================================================================
    # STEP 2: BUILD MODEL
    # ========================================================================
    
    print(f"\n{'STEP 2: BUILDING MODEL':^80}")
    print("=" * 80)
    
    model = ClinicalDecisionSupportModel(
        num_classes=data_loader.num_classes,
        image_size=IMAGE_SIZE,
        backbone=BACKBONE,
        model_dir='models',
        log_dir='logs'
    )
    
    model.build_model()
    model.compile_model()
    
    print(f"\nModel Summary:")
    model.model.summary()
    
    # ========================================================================
    # STEP 3: TRAIN MODEL
    # ========================================================================
    
    print(f"\n{'STEP 3: TRAINING MODEL':^80}")
    print("=" * 80)
    
    history = model.train(
        train_gen,
        val_gen,
        epochs=EPOCHS_PHASE1,
        fine_tune_epochs=EPOCHS_PHASE2
    )
    
    # ========================================================================
    # STEP 4: EVALUATE MODEL
    # ========================================================================
    
    print(f"\n{'STEP 4: EVALUATING MODEL':^80}")
    print("=" * 80)
    
    results, predictions, y_true = model.evaluate(test_gen, data_loader.pathologies)
    
    # ========================================================================
    # STEP 5: SAVE RESULTS
    # ========================================================================
    
    print(f"\n{'STEP 5: SAVING RESULTS':^80}")
    print("=" * 80)
    
    model.plot_training_history('training_history.png')
    model.plot_auc_scores(results, 'auc_scores.png')
    model.save_model('chest_xray_diagnosis_model.h5')
    
    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    
    print(f"\n{'=' * 80}")
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"\nFinal Overall AUC: {results['overall_auc']:.4f}")
    print("\nFiles saved:")
    print("  - training_history.png")
    print("  - auc_scores.png")
    print("  - chest_xray_diagnosis_model.h5")
    print("\nTo view TensorBoard logs, run:")
    print(f"  tensorboard --logdir=logs")
    print("=" * 80)
    
    return model, results


if __name__ == "__main__":
    # Enable GPU memory growth
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print(f"Found {len(physical_devices)} GPU(s)\n")
    else:
        print("No GPU found, using CPU\n")
    
    # Run main pipeline
    model, results = main()

In [ ]:
print('Hello')